In [ ]:
import wavelet2DT_GPU as wt
import wavelet2DT_GPU as hwt
import cwt2D1T as iwt
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import scipy
import importlib
importlib.reload(iwt)
%matplotlib inline

In [ ]:
nx, ny, nz = 25, 30, 35

x = np.arange(nx)
y = np.arange(ny)
z = np.arange(nz)
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')

signal = np.zeros((nx, ny, nz))

fx, fy, fz = 0.2, 0.1, 0.06
sine = np.sin(2 * np.pi * fx * X + 2 * np.pi * fy * Y - 2 * np.pi * fz * Z)
signal += 1.0 * sine

np.random.seed(42)
noise = np.random.normal(0, 0.2, signal.shape)
signal += noise

In [ ]:
fig, ax = plt.subplots()
im = ax.imshow(signal[:, :, 0], origin='lower', cmap='coolwarm', vmin=signal.min(), vmax=signal.max())
def update(frame):
    im.set_data(signal[:, :, frame])
    return [im]
ani = FuncAnimation(fig, update, frames=signal.shape[-1], interval=100, blit=True)
ani.save('./movie/test_signal.gif', dpi=300)

In [ ]:
lambda_r_arr = 1/np.sort(1/np.linspace(-15, 15, 20))
lambda_t_arr = 1/np.sort(1/np.linspace(-20, 20, 30))
period_arr = 1/np.sort(1/np.linspace(2, 25, 16))

kx_array = np.sort(2*np.pi/lambda_r_arr)
ky_array = np.sort(2*np.pi/lambda_t_arr)
omega_array = np.sort(2*np.pi/period_arr)
# w = iwt.cwt_2d1t(signal, kx_array, ky_array, omega_array ,1, 1, 1, k0=-6, epsilon=1)
w = hwt.cwt2DT(scipy.fft.fftn(signal), 1, 1, 1, lambda_r_arr, lambda_t_arr, period_arr)

In [ ]:
signal_rec = -1*iwt.icwt_2d1t(w, kx_array, ky_array, omega_array ,1, 1, 1, k0=-6, omega0=6, epsilon=1)

In [ ]:
fig, ax = plt.subplots()
im = ax.imshow(signal_rec[:, :, 0], origin='lower', cmap='coolwarm', vmin=signal_rec.min(), vmax=signal_rec.max())
def update(frame):
    im.set_data(signal_rec[:, :, frame])
    return [im]
ani = FuncAnimation(fig, update, frames=signal_rec.shape[-1], interval=100, blit=True)
ani.save('./movie/test_signal_rec.gif', dpi=300)

In [ ]:
plt.scatter(signal.flatten(), signal_rec.flatten(), s=1, alpha=0.1)

In [ ]:
# w = wt.cwt2DT(scipy.fft.fftn(signal), 1, 1, 1, 2*np.pi/kx_array, 2*np.pi/ky_array, 2*np.pi/omega_array)
psd = np.mean(np.abs(w)**2, axis=(0, 1, 2))
psd = np.trapezoid(psd, x=ky_array, axis=1)
kx_grid, omega_grid = np.meshgrid(kx_array, omega_array, indexing='ij')

x = kx_grid.ravel()
y = omega_grid.ravel()
z = np.log10(psd).ravel()

fig, ax = plt.subplots(figsize=(6, 5))
mesh = ax.tricontourf(x, y, z, levels=50, cmap='jet')

ax.set_aspect('auto')
ax.set_ylabel(r'$\omega/s^{-1}$')
ax.set_xlabel(r'$k_r/Mm^{-1}$')
ax.grid(True, alpha=0.3)
plt.colorbar(mesh, ax=ax, label=r'$log_{10}(PSD)$')